# THE WORST TRADE OF ALL TIME

Find the most lopsided trade based on **total career value given away**.

This measures what each side GAVE AWAY and calculates the full career points those assets scored from the trade date forward, regardless of who owned them after the trade.

## Methodology

This analysis uses the "hindsight is 20/20" approach:
- For **players**: Counts ALL career fantasy points scored from the trade date forward
- For **draft picks**: Counts ALL career points of the drafted player
- **Does NOT** filter by subsequent ownership - we measure the true long-term value lost

This answers: "What was the worst trade when evaluated by pure asset value given away?"

In [ ]:
spark.sql("USE CATALOG workspace")

## Top 5 Worst Trades

In [ ]:
top_worst_trades = spark.sql("""
SELECT
  transaction_id,
  trade_season,
  roster_a,
  roster_b,
  roster_a_manager,
  roster_b_manager,
  ROUND(roster_a_points, 1) as roster_a_points,
  ROUND(roster_b_points, 1) as roster_b_points,
  ROUND(roster_a_vor, 1) as roster_a_vor,
  ROUND(roster_b_vor, 1) as roster_b_vor,
  ROUND(point_differential, 1) as point_differential,
  ROUND(vor_differential, 1) as vor_differential,
  points_winner_manager,
  points_loser_manager,
  vor_winner_manager,
  vor_loser_manager,
  ROUND(trade_impact_magnitude_points, 1) as trade_impact_magnitude_points,
  ROUND(trade_impact_magnitude_vor, 1) as trade_impact_magnitude_vor,
  winners_differ,
  trade_is_complete
FROM workspace.sleeper_trades.agg_trade_winners_enriched
WHERE cluster_name = 'League of Inches'
  AND trade_is_complete = TRUE
ORDER BY trade_impact_magnitude_vor DESC
LIMIT 5
""")

display(top_worst_trades)

## THE Worst Trade - Complete Breakdown

Shows what each side GAVE AWAY and the career points lost.

In [ ]:
# Get complete asset breakdown for the worst trade
worst_trade_breakdown = spark.sql(f"""
WITH worst_trade AS (
  SELECT 
    transaction_id, 
    trade_season,
    roster_a,
    roster_b,
    roster_a_manager,
    roster_b_manager
  FROM workspace.sleeper_trades.agg_trade_winners_enriched
  WHERE cluster_name = 'League of Inches'
    AND trade_is_complete = TRUE
  ORDER BY trade_impact_magnitude_vor DESC
  LIMIT 1
),
-- Get RECEIVED players with their career points and VOR
received_players AS (
  SELECT
    wt.transaction_id,
    fa.side_roster_id,
    CASE 
      WHEN fa.side_roster_id = wt.roster_a THEN wt.roster_b_manager
      ELSE wt.roster_a_manager
    END as manager_gave_away,
    'player' as asset_type,
    p.full_name as asset_name,
    p.position,
    COALESCE(SUM(pp.points), 0) as total_career_points,
    COALESCE(SUM(pp.vor), 0) as total_career_vor
  FROM worst_trade wt
  JOIN sleeper_trades.fact_trade_player_assets fa
    ON wt.transaction_id = fa.transaction_id
    AND fa.direction = 'incoming'
  JOIN sleeper_core.dim_players p ON fa.player_id = p.player_id
  LEFT JOIN sleeper_trades.fact_trade_player_points_multi_season pp
    ON fa.transaction_id = pp.transaction_id
    AND fa.side_roster_id = pp.side_roster_id
    AND fa.player_id = pp.player_id
  GROUP BY wt.transaction_id, fa.side_roster_id, wt.roster_a, wt.roster_b, 
           wt.roster_a_manager, wt.roster_b_manager, p.full_name, p.position
),
-- Get RECEIVED picks with career points and VOR - join to bridge and career points
received_picks AS (
  SELECT
    wt.transaction_id,
    fpa.side_roster_id,
    CASE
      WHEN fpa.side_roster_id = wt.roster_a THEN wt.roster_b_manager
      ELSE wt.roster_a_manager
    END as manager_gave_away,
    'pick' as asset_type,
    CASE
      WHEN bp.player_id IS NOT NULL THEN CONCAT(fpa.pick_season, ' Round ', fpa.round, ' (', p.full_name, ')')
      ELSE CONCAT(fpa.pick_season, ' Round ', fpa.round, ' (unrealized)')
    END as asset_name,
    p.position,
    COALESCE(SUM(pp.points), 0) as total_career_points,
    COALESCE(SUM(pp.vor), 0) as total_career_vor
  FROM worst_trade wt
  JOIN sleeper_trades.fact_trade_pick_assets fpa
    ON wt.transaction_id = fpa.transaction_id
  LEFT JOIN sleeper_trades.bridge_trade_pick_to_player bp
    ON fpa.league_id = bp.league_id
    AND fpa.transaction_id = bp.transaction_id
    AND fpa.side_roster_id = bp.side_roster_id
    AND fpa.pick_season = bp.pick_season
    AND fpa.round = bp.round
  LEFT JOIN sleeper_trades.fact_trade_pick_points_career pp
    ON bp.transaction_id = pp.transaction_id
    AND bp.side_roster_id = pp.side_roster_id
    AND bp.player_id = pp.player_id
  LEFT JOIN sleeper_core.dim_players p ON bp.player_id = p.player_id
  GROUP BY wt.transaction_id, fpa.side_roster_id, wt.roster_a, wt.roster_b,
           wt.roster_a_manager, wt.roster_b_manager, fpa.pick_season, fpa.round,
           bp.player_id, p.full_name, p.position
)
SELECT
  manager_gave_away as manager_name,
  asset_type,
  asset_name,
  position,
  ROUND(total_career_points, 1) as career_points_given_away,
  ROUND(total_career_vor, 1) as career_vor_given_away
FROM received_players
UNION ALL
SELECT
  manager_gave_away as manager_name,
  asset_type,
  asset_name,
  position,
  ROUND(total_career_points, 1) as career_points_given_away,
  ROUND(total_career_vor, 1) as career_vor_given_away
FROM received_picks
ORDER BY manager_name, asset_type DESC, career_vor_given_away DESC
""")

display(worst_trade_breakdown)

## Summary by Manager

In [ ]:
from pyspark.sql import functions as F

# Calculate total career points and VOR given away by each manager
manager_totals = worst_trade_breakdown.groupBy('manager_name').agg(
    F.sum('career_points_given_away').alias('total_career_points_given_away'),
    F.sum('career_vor_given_away').alias('total_career_vor_given_away')
).orderBy(F.desc('total_career_vor_given_away'))

display(manager_totals)

# Extract variables for visualizations
worst_trade_info = top_worst_trades.first()
vor_loser = worst_trade_info['vor_loser_manager']
vor_winner = worst_trade_info['vor_winner_manager']
points_loser = worst_trade_info['points_loser_manager']
points_winner = worst_trade_info['points_winner_manager']
worst_trade_season = worst_trade_info['trade_season']
magnitude_vor = worst_trade_info['trade_impact_magnitude_vor']
magnitude_points = worst_trade_info['trade_impact_magnitude_points']
winners_differ = worst_trade_info['winners_differ']
worst_trade_id = worst_trade_info['transaction_id']

## Visualization

In [ ]:
import matplotlib.pyplot as plt

# Convert to pandas for plotting
pdf = manager_totals.toPandas()

if not pdf.empty:
    # Create side-by-side comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Points-based visualization
    bars1 = ax1.bar(pdf['manager_name'], pdf['total_career_points_given_away'])
    for i, bar in enumerate(bars1):
        if pdf.iloc[i]['manager_name'] == points_loser:
            bar.set_color('red')
        else:
            bar.set_color('green')
    
    ax1.set_xlabel('Manager', fontsize=12)
    ax1.set_ylabel('Total Career Points Given Away', fontsize=12)
    ax1.set_title(f'Raw Points Analysis\nDifferential: {magnitude_points} points', 
                 fontsize=12, fontweight='bold')
    ax1.tick_params(axis='x', rotation=45)
    
    # VOR-based visualization
    bars2 = ax2.bar(pdf['manager_name'], pdf['total_career_vor_given_away'])
    for i, bar in enumerate(bars2):
        if pdf.iloc[i]['manager_name'] == vor_loser:
            bar.set_color('red')
        else:
            bar.set_color('green')
    
    ax2.set_xlabel('Manager', fontsize=12)
    ax2.set_ylabel('Total Career VOR Given Away', fontsize=12)
    ax2.set_title(f'Value Over Replacement (VOR) Analysis\nDifferential: {magnitude_vor} VOR', 
                 fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    
    # Overall title
    fig.suptitle(f'THE WORST TRADE OF ALL TIME - Season {worst_trade_season}\n{"⚠️ WINNERS DIFFER! ⚠️" if winners_differ else "Winners Agree"}', 
                 fontsize=16, fontweight='bold', color='red' if winners_differ else 'black')
    
    plt.tight_layout()
    display(fig)
else:
    print("No data to plot")

## Asset Breakdown Visualization

In [ ]:
import matplotlib.pyplot as plt

pdf_assets = worst_trade_breakdown.toPandas()

if not pdf_assets.empty:
    # Get unique managers
    managers = pdf_assets['manager_name'].unique()
    
    # Create side-by-side comparison for each manager (points vs VOR)
    fig, axes = plt.subplots(2, len(managers), figsize=(15, 12), sharey='row')
    
    if len(managers) == 1:
        axes = axes.reshape(-1, 1)
    
    for idx, manager in enumerate(managers):
        manager_data = pdf_assets[pdf_assets['manager_name'] == manager]
        
        # Sort by VOR for consistent ordering
        manager_data = manager_data.sort_values('career_vor_given_away', ascending=True)
        
        # Color code by asset type
        colors = ['#FF6B6B' if asset_type == 'player' else '#4ECDC4' 
                  for asset_type in manager_data['asset_type']]
        
        # Top row: Points
        axes[0, idx].barh(manager_data['asset_name'], manager_data['career_points_given_away'], color=colors)
        axes[0, idx].set_xlabel('Career Points', fontsize=10)
        
        # Add border if this is the points loser
        if manager == points_loser:
            axes[0, idx].set_title(f'{manager}\n(Points LOSER)', fontsize=11, fontweight='bold', color='red')
            for spine in axes[0, idx].spines.values():
                spine.set_edgecolor('red')
                spine.set_linewidth(2)
        else:
            axes[0, idx].set_title(f'{manager}\n(Points Winner)', fontsize=11, fontweight='bold', color='green')
        
        # Bottom row: VOR
        axes[1, idx].barh(manager_data['asset_name'], manager_data['career_vor_given_away'], color=colors)
        axes[1, idx].set_xlabel('Career VOR', fontsize=10)
        
        # Add border if this is the VOR loser
        if manager == vor_loser:
            axes[1, idx].set_title(f'{manager}\n(VOR LOSER)', fontsize=11, fontweight='bold', color='red')
            for spine in axes[1, idx].spines.values():
                spine.set_edgecolor('red')
                spine.set_linewidth(2)
        else:
            axes[1, idx].set_title(f'{manager}\n(VOR Winner)', fontsize=11, fontweight='bold', color='green')
        
        axes[0, idx].tick_params(axis='y', labelsize=8)
        axes[1, idx].tick_params(axis='y', labelsize=8)
    
    axes[0, 0].set_ylabel('Asset', fontsize=10)
    axes[1, 0].set_ylabel('Asset', fontsize=10)
    
    plt.suptitle(f'Assets Given Away - THE WORST TRADE\nTransaction: {worst_trade_id}\n{"⚠️ WINNERS DIFFER BETWEEN POINTS AND VOR! ⚠️" if winners_differ else ""}', 
                 fontsize=14, fontweight='bold', color='red' if winners_differ else 'black')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#FF6B6B', label='Player'),
        Patch(facecolor='#4ECDC4', label='Draft Pick')
    ]
    fig.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    display(fig)
else:
    print("No asset data to plot")

---

## Analysis Complete

This notebook reveals:
1. The top 5 worst trades by **VOR differential** (scarcity-adjusted value)
2. Complete breakdown of THE worst trade showing **both raw points AND VOR**
3. What each side gave away and the career impact in both metrics
4. Visual comparison showing if the winner differs between points and VOR analysis

**Key Insights:**
- **Raw Points:** Traditional fantasy scoring without positional adjustment
- **VOR (Value Over Replacement):** Accounts for positional scarcity
  - Trading away a top QB matters less than trading away a top RB/WR/TE
  - VOR reveals the TRUE value lost when considering draft/waiver alternatives
- **Winners Differ Flag:** Highlights trades where positional scarcity changed the outcome

**Methodology:** This calculates the full career value of assets given away, regardless of who owned them after the trade. It's the "hindsight is 20/20" worst trade analysis with both traditional and scarcity-adjusted metrics.